# Baseline classifier — separate the 3 noise types from raw CB data

Turn each CB run (one parquet = one example) into a few variance-based features,
then a simple logistic-regression model + 5-fold stratified CV.
This is the **number diffusion has to beat**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics         import confusion_matrix, ConfusionMatrixDisplay

In [2]:
# load all 450 examples of raw CB data
DATA_DIR = Path("data")
files = sorted(DATA_DIR.glob("*.parquet"))
df_all = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"loaded files: {len(files)} and total rows is {len(df_all)}")

# per example spread: var across 30 l values but per m value
keys = ["noisetype","param_value", "seed", "P", "m"]
var_Pm = df_all.groupby(keys)["f_pml"].var().reset_index(name="var_Pm")
print(var_Pm.shape)
display(var_Pm.tail())

def slope(g):
    return np.polyfit(g["m"], g["var_Pm"], 1)[0]

ex_keys = ["noisetype", "param_value", "seed"]
slopes = (var_Pm.groupby(ex_keys + ["P"])).apply(slope, include_groups=False).reset_index(name="slope_P")
print(slopes.shape)
display(slopes.head())
display(slopes.tail())

# step (c): one row per example; slope_X, slope_Y, slope_Z become columns
feat = slopes.pivot_table(index=ex_keys, columns="P", values="slope_P").reset_index()
print(feat.shape)      # expect 450 rows
display(feat.head())
display(feat.tail())


loaded files: 450 and total rows is 243000
(8100, 6)


,noisetype,param_value,seed,P,m,var_Pm
8095,memory,0.2,29,Z,4,0.046598
8096,memory,0.2,29,Z,6,0.131483
8097,memory,0.2,29,Z,8,0.224598
8098,memory,0.2,29,Z,12,0.257434
8099,memory,0.2,29,Z,16,0.244569


(1350, 5)


,noisetype,param_value,seed,P,slope_P
0,drift,0.05,0,X,0.000000
1,drift,0.05,0,Y,0.000076
2,drift,0.05,0,Z,0.000038
3,drift,0.05,1,X,0.000000
4,drift,0.05,1,Y,0.000093


,noisetype,param_value,seed,P,slope_P
1345,memory,0.2,28,Y,0.016274
1346,memory,0.2,28,Z,0.003218
1347,memory,0.2,29,X,0.016447
1348,memory,0.2,29,Y,0.009275
1349,memory,0.2,29,Z,0.016624


(450, 6)


P,noisetype,param_value,seed,X,Y,Z
0,drift,0.05,0,0.0,0.000076,0.000038
1,drift,0.05,1,0.0,0.000093,0.000037
2,drift,0.05,2,0.0,0.000089,0.000051
3,drift,0.05,3,0.0,0.000050,0.000078
4,drift,0.05,4,0.0,0.000045,0.000027


P,noisetype,param_value,seed,X,Y,Z
445,memory,0.2,25,0.012170,0.012563,0.009738
446,memory,0.2,26,0.013450,0.007643,0.011090
447,memory,0.2,27,0.011354,0.008042,0.013178
448,memory,0.2,28,0.010231,0.016274,0.003218
449,memory,0.2,29,0.016447,0.009275,0.016624


In [3]:
feature_cols = ["X", "Y","Z"]
Xmat = feat[feature_cols].to_numpy(dtype=float)
y = feat["noisetype"].to_numpy()
# scale + logistic regression, scored by 5-fold stratified CV
pipeline    = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pipeline, Xmat, y, cv=splitter)
print("fold accuracies:", scores.round(3))
print(f"mean accuracy:{scores.mean():.3f} +/- {scores.std():.3f}")

fold accuracies: [0.911 0.9   0.867 0.9   0.856]
mean accuracy:0.887 +/- 0.022
